# Label-efficient lung nodule segmentation: 3D Mamba-UNet + diffusion pretraining (LIDC-IDRI)

**Kaggle setup:** *Settings → Accelerator: GPU T4 x1 (or P100)*, *Internet: on*, and *Add Input →* the dataset
**`justinkirby/the-cancer-imaging-archive-lidcidri`** (the TCIA LIDC-IDRI mirror, full DICOM).

This notebook runs the whole study:
1. It preprocesses every LIDC nodule annotated by ≥2 radiologists into 96³ cubes at 0.8 mm, using the 50% consensus mask plus each reader's own mask.
2. It pretrains the network as a diffusion denoiser on unlabelled CT cubes.
3. It fine-tunes {UNet, Mamba-UNet, Mamba-UNet+pretraining} on 10/25/50/100% of the labelled training patients.
4. It tests every model on the same held-out patients and compares against radiologist inter-reader agreement.

Runs that already finished are skipped, so you can re-run the notebook after a timeout.

In [ ]:
!git clone -q https://github.com/chrisjason0505/lung-nodule-segmentation-mamba-diffusion.git repo || (cd repo && git pull -q)
%cd repo
!pip install -q "setuptools<81" pylidc==0.2.3 mambapy pydicom
import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

Optional: the fused `mamba-ssm` CUDA kernels are about 3–5× faster than the pure-PyTorch scan. The install can take a while,
and if it fails the code falls back to the PyTorch backend automatically. Both backends compute the same selective scan.

In [ ]:
INSTALL_MAMBA_SSM = True
if INSTALL_MAMBA_SSM:
    !pip install -q causal-conv1d mamba-ssm --no-build-isolation 2>&1 | tail -2
!python -c "from lungseg.models.ssm import backend_name; print('Mamba backend:', backend_name())"

In [ ]:
!python -m pytest -q tests   # checks the scan maths, the model, the metrics, and image/mask alignment in preprocessing

In [ ]:
# the indexer walks everything under this folder and finds the LIDC CT series by SeriesInstanceUID,
# so the exact mount path of the Kaggle dataset doesn't matter
DICOM_ROOT = '/kaggle/input'
DATA = '/kaggle/working/lidc_crops'
RUNS = '/kaggle/working/runs'
!ls /kaggle/input

In [ ]:
# about 1–2 h for all 1,018 scans on Kaggle's 4 CPUs (only the slices near each nodule are decoded)
!python -m lungseg.data.prepare_lidc --dicom_root "$DICOM_ROOT" --out "$DATA" --workers 4 --min_readers 2

In [ ]:
# the full study (3 configs x 4 label fractions + 1 pretraining run)
# lower --steps / --pretrain_steps, or use --preset mamba_only, if you are short on GPU time
!python -m lungseg.experiments --data "$DATA" --runs "$RUNS" --preset core --steps 5000 --pretrain_steps 6000

In [ ]:
from IPython.display import Image, Markdown, display
display(Markdown(open(f'{RUNS}/summary.md').read()))
display(Image(f'{RUNS}/data_efficiency.png'))

In [ ]:
best = f'{RUNS}/mamba_ddep_f1_s0/best.pt'
!python -m lungseg.evaluate --data "$DATA" --ckpt "$best"
!python -m lungseg.visualize --data "$DATA" --ckpt "$best" --out /kaggle/working/predictions.png
display(Image('/kaggle/working/predictions.png'))